<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/create_knowledge_base/create_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================

!pip install -q sentence-transformers huggingface_hub tqdm


In [3]:
# OPTION 1:
# If files are directly in a GitHub repo,
# easiest approach is git clone.

!git clone https://github.com/anirbanghoshsbi/create_knowledge_base.git


Cloning into 'create_knowledge_base'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 95 (delta 43), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 81.44 KiB | 11.63 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [6]:


# =========================================================
# IMPORTS
# =========================================================

import json
import os
from pathlib import Path
from tqdm import tqdm

from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer


# =========================================================
# CONFIG
# =========================================================

# Replace with your repo details
REPO_ID = "anirbanghoshsbi/create_knowledge_base"

# Folder inside repo containing JSON files
REPO_FOLDER = "json_files"

# Number of files to process
MAX_FILES = 20

# Output directory
OUTPUT_DIR = "kb_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# DOWNLOAD JSON FILES FROM GITHUB/HF REPO
# =========================================================



# =========================================================
# FIND JSON FILES
# =========================================================

repo_path = Path("create_knowledge_base")

json_files = sorted(list(repo_path.rglob("*.json")))[:MAX_FILES]

print(f"Found {len(json_files)} JSON files")


# =========================================================
# LOAD + ADD UNIQUE IDS
# =========================================================

master_concepts = []

counter = 1

for file_path in tqdm(json_files):

    try:

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

    except Exception as e:

        print(f"\nERROR reading: {file_path}")
        print(e)

        continue

    # -------------------------------
    # CASE 1: LIST OF CONCEPTS
    # -------------------------------

    if isinstance(data, list):

        for concept in data:

            concept["concept_id"] = f"PM_{counter:05d}"

            concept["source_file"] = str(file_path.name)

            master_concepts.append(concept)

            counter += 1

    # -------------------------------
    # CASE 2: DICT WITH "concepts"
    # -------------------------------

    elif isinstance(data, dict) and "concepts" in data:

        for concept in data["concepts"]:

            concept["concept_id"] = f"PM_{counter:05d}"

            concept["source_file"] = str(file_path.name)

            master_concepts.append(concept)

            counter += 1

    else:

        print(f"\nSkipping unsupported format: {file_path}")

print(f"\nTotal concepts merged: {len(master_concepts)}")

# =========================================================
# SAVE MASTER JSON
# =========================================================

master_json_path = os.path.join(
    OUTPUT_DIR,
    "master_concepts.json"
)

with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(master_concepts, f, indent=2, ensure_ascii=False)

print(f"\nSaved master JSON to:\n{master_json_path}")


# =========================================================
# CREATE TEXT DATASET FOR EMBEDDINGS
# =========================================================

embedding_dataset = []

for concept in master_concepts:

    title = concept.get("title", "")
    concept_type = concept.get("type", "")
    statement = concept.get("concise_statement", "")
    explanation = concept.get("detailed_explanation", "")

    implications = concept.get("implications", [])

    implications_text = " ".join(implications)

    # --------------------------------------------
    # TEMPORARY SEMANTIC TEXT FOR EMBEDDING MODEL
    # --------------------------------------------

    embedding_text = f"""
    Title: {title}

    Type: {concept_type}

    Statement:
    {statement}

    Explanation:
    {explanation}

    Implications:
    {implications_text}
    """

    embedding_dataset.append({
        "concept_id": concept["concept_id"],
        "text_for_embedding": embedding_text.strip()
    })


# =========================================================
# SAVE TEXT DATASET
# =========================================================

embedding_text_path = os.path.join(
    OUTPUT_DIR,
    "embedding_text_dataset.json"
)

with open(embedding_text_path, "w", encoding="utf-8") as f:
    json.dump(embedding_dataset, f, indent=2, ensure_ascii=False)

print(f"\nSaved embedding text dataset to:\n{embedding_text_path}")


# =========================================================
# LOAD EMBEDDING MODEL
# =========================================================

model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("\nEmbedding model loaded")


# =========================================================
# CREATE EMBEDDINGS
# =========================================================

texts = [
    x["text_for_embedding"]
    for x in embedding_dataset
]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)


# =========================================================
# SAVE EMBEDDINGS
# =========================================================

embedding_output = []

for i, item in enumerate(embedding_dataset):

    embedding_output.append({
        "concept_id": item["concept_id"],
        "embedding": embeddings[i].tolist()
    })


embedding_save_path = os.path.join(
    OUTPUT_DIR,
    "concept_embeddings.json"
)

with open(embedding_save_path, "w", encoding="utf-8") as f:
    json.dump(embedding_output, f)

print(f"\nSaved embeddings to:\n{embedding_save_path}")


# =========================================================
# QUICK TEST
# =========================================================

print("\nSample embedding vector length:")
print(len(embedding_output[0]["embedding"]))

print("\nSample concept ID:")
print(embedding_output[0]["concept_id"])



Found 20 JSON files


100%|██████████| 20/20 [00:00<00:00, 4265.54it/s]


ERROR reading: create_knowledge_base/same_as_ever/part01.json
Expecting value: line 1 column 1 (char 0)

Total concepts merged: 189

Saved master JSON to:
kb_output/master_concepts.json

Saved embedding text dataset to:
kb_output/embedding_text_dataset.json


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Saved embeddings to:
kb_output/concept_embeddings.json

Sample embedding vector length:
768

Sample concept ID:
PM_00001
